<a href="https://colab.research.google.com/github/mirzafani808-source/neurofive-ml-track/blob/main/titanic-survival-prediction/titanic_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!wget -O titanic.csv https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv


--2026-07-21 09:47:06--  https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60302 (59K) [text/plain]
Saving to: ‘titanic.csv’

titanic.csv         100%[===================>]  58.89K  --.-KB/s    in 0.04s   

2026-07-21 09:47:07 (1.42 MB/s) - ‘titanic.csv’ saved [60302/60302]



In [3]:
"""
Predict Titanic Survival — First Classification Model
--------------------------------------------------------
Loads the cleaned Titanic dataset, encodes categorical features,
splits into train/test sets, trains a Logistic Regression model,
and evaluates it with accuracy and a confusion matrix.
"""

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------
df = pd.read_csv("titanic.csv")

# ---------------------------------------------------------
# 2. Clean data (this should already be mostly done from your EDA step)
# ---------------------------------------------------------
# Fill missing Age with the median age
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill missing Embarked with the most common port
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Drop columns that aren't useful for a first model
# (Name/Ticket/Cabin are high-cardinality/free text; PassengerId is just an index)
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# ---------------------------------------------------------
# 3. Encode categorical columns
# ---------------------------------------------------------
df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)
# drop_first=True avoids the "dummy variable trap" (redundant columns)

# ---------------------------------------------------------
# 4. Split features/target and train/test sets
# ---------------------------------------------------------
X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# 5. Train Logistic Regression model
# ---------------------------------------------------------
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# ---------------------------------------------------------
# 6. Evaluate
# ---------------------------------------------------------
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc:.4f}\n")

print("Confusion Matrix:")
print(cm)
print()
print("                Predicted: Died   Predicted: Survived")
print(f"Actual: Died        {cm[0][0]:>5}              {cm[0][1]:>5}")
print(f"Actual: Survived     {cm[1][0]:>5}              {cm[1][1]:>5}")
print()

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Died", "Survived"]))

# ---------------------------------------------------------
# 7. Feature coefficients (bonus: which features matter most)
# ---------------------------------------------------------
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", key=abs, ascending=False)

print("Feature importance (Logistic Regression coefficients):")
print(coef_df.to_string(index=False))


Accuracy: 0.8045

Confusion Matrix:
[[98 12]
 [23 46]]

                Predicted: Died   Predicted: Survived
Actual: Died           98                 12
Actual: Survived        23                 46

Classification Report:
              precision    recall  f1-score   support

        Died       0.81      0.89      0.85       110
    Survived       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

Feature importance (Logistic Regression coefficients):
   Feature  Coefficient
  Sex_male    -2.555755
    Pclass    -1.090476
Embarked_S    -0.381616
Embarked_Q     0.278320
     SibSp    -0.243809
     Parch    -0.071496
       Age    -0.038464
      Fare     0.002255
